# Root-Cause Investigation of Degraded Cities

Identifying operational behaviours associated with rider-experience degradation.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.root_cause import (
    identify_degraded_cities,
    compare_degraded_vs_stable,
    trace_operational_chain,
    assess_city_consistency,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# Identify degraded cities
degraded = identify_degraded_cities(df)
print('Degraded cities:')
for city in degraded:
    print(f'  {city.city}: score={city.deterioration_score:.2f}')
    for metric, value in city.metrics.items():
        print(f'    {metric}: {value:.3f}')

In [ ]:
# Compare degraded vs stable
degraded_cities = [c.city for c in degraded[:2]]
comparisons = compare_degraded_vs_stable(df, degraded_cities)
for comp in comparisons:
    print(f'\n{comp.category}:')
    print(f'  Cities: {comp.cities}')
    print(f'  Avg values: {comp.avg_values}')
    print(f'  Deterioration: {comp.deterioration}')

In [ ]:
# Operational chain
links = trace_operational_chain(df)
print('Operational chain:')
for link in links:
    print(f'  {link.from_factor} → {link.to_factor}: {link.correlation:.3f} ({link.strength})')

In [ ]:
# City consistency
consistency = assess_city_consistency(df)
print('City consistency (demand/supply vs cancel):')
for city, metrics in consistency.items():
    print(f'  {city}: {metrics}')